# Sequenzmodelle (LSTM/GRU) neu bewerten

**Autor:** jd · **Datum:** 2026-10-20

**Ziel:** Den überarbeiteten Sequenz-Klassifikator (#173) fair gegen die kombinierte SVM vergleichen (#174). Die bisherigen LSTMs (Notebooks 19–27) kamen auf höchstens 0,700 BAcc kombiniert. Ihr Training endete aber am Epochen-Limit, ohne dass das frühe Stoppen griff.

**Hypothese:** Mit GRU/LSTM über Bandblöcke, Wellenlängen-Kanal, MLP-Kopf und ausreichend Epochen lernt das Netz deutlich mehr aus dem Spektrum. Die SVM (CV 0,859) schlägt es bei rund 4.500 Trainingszeilen trotzdem nicht.

**Vorgehen:** Alle Konfigurationen werden mit `reduction_curve` per CV auf dem 80/20-Split bewertet (Kultur, Stadium, kombiniert je Fold). AEZ und Monat gehen als gelernte Embeddings ein. Danach wird das beste Sequenzmodell **einmal** validiert.

| Konfiguration | Eingabe | Was sie zeigt |
| --- | --- | --- |
| SVM | alle Bänder | Referenz (beste Einstellung aus #155) |
| LSTM alt | alle Bänder, 1 Band je Schritt, linearer Kopf, 50 Epochen | bisheriger Stand, jetzt auf 80/20 |
| LSTM/GRU neu | alle Bänder, Bandblöcke + Wellenlänge + MLP-Kopf | Wirkung der Überarbeitung |
| LSTM/GRU neu, Sentinel-2 | 10 Sensorbänder | kurze Sequenz, Bezug zu #155 |
| beste neue ohne Kontext | nur Spektrum | was das Netz aus dem Spektrum allein lernt |

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.base import BaseEstimator

from awp2.config import (
    LSTM_EMBEDDINGS_CONTEXT,
    LSTM_NO_CONTEXT,
    MODEL_DOCS_FIGURES_DIR,
    REDUCTION_BEST_PARAMS_COL,
    SENTINEL2_SENSOR,
)
from awp2.evaluation import METRIC_LABELS
from awp2.experiment import RunConfig, TuneConfig, run
from awp2.models import make_combined_spectral_lstm
from awp2.models.combined import make_combined_svm
from awp2.plots import plot_metric_comparison, save_doc_figure
from awp2.preprocessing import PreprocessingConfig
from awp2.reduction import CURVE_METRICS, ReductionVariant, reduction_curve

# the old configuration stops at max_epochs on purpose; its warning is expected here
warnings.filterwarnings("ignore", message="Training stopped at max_epochs")

## 1. Konfigurationen

Das Raster für die neuen Modelle bleibt klein: Zelltyp, Bänder je Schritt und Lernrate. Breite (64), MLP-Kopf (64), Dropout (0,2) und bis zu 200 Epochen mit frühem Stoppen sind fest.

In [ ]:
APPROACH = "combined"
spectra = PreprocessingConfig(use_spectral_standard_scale=True)
embedded = spectra.model_copy(update={"use_categorical_context": True})
sentinel2 = embedded.model_copy(update={"spectral_sensor": SENTINEL2_SENSOR})
no_context = spectra.model_copy(update={"use_meta": False})

fixed_new = {
    "estimator__hidden_size": [64],
    "estimator__num_layers": [1],
    "estimator__dropout": [0.2],
    "estimator__head_hidden_size": [64],
    "estimator__wavelength_channel": [True],
    "estimator__max_epochs": [200],
}
configs = {
    "svm": (
        make_combined_svm(),
        spectra,
        {"estimator__C": [300.0], "estimator__gamma": [0.003]},
    ),
    "lstm_old": (
        make_combined_spectral_lstm(LSTM_EMBEDDINGS_CONTEXT),
        embedded,
        {
            "estimator__hidden_size": [64],
            "estimator__num_layers": [1],
            "estimator__dropout": [0.2],
            "estimator__max_epochs": [50],
        },
    ),
    "sequence_new": (
        make_combined_spectral_lstm(LSTM_EMBEDDINGS_CONTEXT),
        embedded,
        {
            **fixed_new,
            "estimator__cell": ["lstm", "gru"],
            "estimator__bands_per_step": [4, 8],
            "estimator__learning_rate": [0.001, 0.003],
        },
    ),
    "sequence_sentinel2": (
        make_combined_spectral_lstm(LSTM_EMBEDDINGS_CONTEXT),
        sentinel2,
        {
            **fixed_new,
            "estimator__cell": ["lstm", "gru"],
            "estimator__bands_per_step": [1],
            "estimator__learning_rate": [0.001, 0.003],
        },
    ),
}

## 2. CV-Vergleich

Jede Konfiguration wird getunt (`crop-stage`-Experiment, approach `combined`), danach wird die beste Einstellung je Fold bewertet. Die SVM und das alte LSTM haben nur einen Kandidaten.

In [ ]:
def evaluate_config(
    name: str,
    model: BaseEstimator,
    preprocessing: PreprocessingConfig,
    grid: dict[str, list[Any]],
) -> pd.DataFrame:
    return reduction_curve(
        model,
        [ReductionVariant(name=name, family="all_bands", preprocessing=preprocessing)],
        TuneConfig(
            name=f"seq_{name}",
            approach=APPROACH,
            description=f"Sequenzmodell-Bewertung (#174): {name}",
            param_grid=grid,
            preprocessing=preprocessing,
        ),
    )


results = pd.concat(
    [evaluate_config(name, *config) for name, config in configs.items()], ignore_index=True
)
results.drop(columns=["family"]).round(3)

## 3. Was lernt das Netz aus dem Spektrum allein?

Die beste neue Einstellung, einmal ohne AEZ und Monat. Die SVM schafft ohne Kontext 0,787 (#155), das alte LSTM kam ohne Kontext auf 0,29.

In [ ]:
best_new = results.set_index("variant").loc["sequence_new", REDUCTION_BEST_PARAMS_COL]
no_context_params = {
    key: [value] for key, value in {**best_new, "estimator__context_mode": LSTM_NO_CONTEXT}.items()
}
ablation = evaluate_config(
    "sequence_no_context",
    make_combined_spectral_lstm(LSTM_NO_CONTEXT),
    no_context,
    no_context_params,
)
results = pd.concat([results, ablation], ignore_index=True)
results[["variant", *CURVE_METRICS, "bacc_combined_std", REDUCTION_BEST_PARAMS_COL]].round(3)

In [ ]:
order = ["svm", "lstm_old", "sequence_new", "sequence_sentinel2", "sequence_no_context"]
labels = {
    "svm": "SVM",
    "lstm_old": "LSTM alt",
    "sequence_new": "LSTM/GRU neu",
    "sequence_sentinel2": "LSTM/GRU neu, S2",
    "sequence_no_context": "neu ohne Kontext",
}
table = results.set_index("variant").loc[order]
fig, ax = plt.subplots(figsize=(10, 4.5))
table[list(CURVE_METRICS)].rename(index=labels, columns=METRIC_LABELS).plot.bar(
    ax=ax,
    yerr=table[[f"{m}_std" for m in CURVE_METRICS]].to_numpy().T,
    rot=0,
    capsize=2,
    width=0.8,
)
ax.set_ylabel("BAcc (CV)")
ax.set_title("Sequenzmodelle vs. SVM")
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize="small", loc="lower left")
save_doc_figure(fig, "sequenzmodelle_cv", MODEL_DOCS_FIGURES_DIR)

**Befund CV:**

| Konfiguration | Kultur | Stadium | kombiniert |
| --- | --- | --- | --- |
| SVM | 0,935 | 0,905 | **0,859** ± 0,025 |
| LSTM alt (wie Notebooks 19–27) | 0,818 | 0,761 | 0,697 ± 0,028 |
| LSTM/GRU neu, alle Bänder | 0,865 | 0,832 | 0,783 ± 0,023 |
| LSTM/GRU neu, Sentinel-2 | 0,884 | 0,856 | **0,794** ± 0,015 |
| neu, ohne Kontext | 0,648 | 0,569 | 0,482 ± 0,024 |

- **Die Überarbeitung wirkt deutlich:** +0,086 gegenüber der alten Konfiguration (0,697 → 0,783). Der alte Wert bestätigt die früheren Läufe (0,711 CV auf 70/30).
- **Das Netz lernt jetzt aus dem Spektrum:** Ohne Kontext steigt es von 0,29 (alt, 70/30) auf 0,48. Die SVM schafft ohne Kontext aber 0,787. Der Großteil der Leistung kommt weiterhin aus den AEZ/Monat-Embeddings.
- **Sentinel-2 ist für das Netz sogar besser als alle Bänder** (0,794 gegenüber 0,783, kleinere Streuung). Mit 10 statt 19 Schritten ist die Sequenz kürzer und leichter zu lernen. Anders als bei der SVM spricht das dafür, dass das Netz unter zu vielen, redundanten Eingaben leidet (Hughes-Phänomen). Für M3 heißt das: Weniger Input nützt hier aktiv.
- **LSTM und GRU sind gleichauf.** Im vollen Spektrum gewinnt LSTM (0,783), bei Sentinel-2 GRU (0,794). Die Abstände der besten Kandidaten liegen unter 0,005 und damit im Rauschen. Am meisten bringen die Bandblöcke: 8 Bänder pro Schritt (19 Schritte) sind besser als 4 (35 Schritte).
- **Die SVM bleibt klar vorn:** 0,065 vor dem besten Sequenzmodell, bei rund 4.500 Trainingszeilen.

## 4. Einmalige Validierung

Das beste Sequenzmodell nach CV (ohne die Ablation ohne Kontext) wird einmal auf dem Validierungsteil bewertet. Die SVM-Referenz auf demselben Split ist `reduction_svm_all_bands` (`dac6bb78`, BAcc kombiniert 0,866).

In [ ]:
candidates = results[results["variant"].isin(["lstm_old", "sequence_new", "sequence_sentinel2"])]
winner = candidates.loc[candidates["bacc_combined"].idxmax()]
winner_preprocessing = {
    "lstm_old": embedded,
    "sequence_new": embedded,
    "sequence_sentinel2": sentinel2,
}[winner["variant"]]
result = run(
    make_combined_spectral_lstm(LSTM_EMBEDDINGS_CONTEXT).set_params(
        **winner[REDUCTION_BEST_PARAMS_COL]
    ),
    RunConfig(
        name=f"combined_{winner['variant']}",
        approach=APPROACH,
        description=f"Bestes Sequenzmodell nach CV (#174): {winner['variant']}",
        tuning_run=winner["tuning_run"],
        preprocessing=winner_preprocessing,
    ),
)
pd.Series(
    {**result.metrics.model_dump(), "run_id": result.run_id[:8], "variant": winner["variant"]}
)

In [ ]:
from awp2.evaluation import Metrics

svm_validation = Metrics(
    bacc_crop=0.9195,
    f1_macro_crop=0.9115,
    bacc_stage=0.9134,
    f1_macro_stage=0.9136,
    bacc_combined=0.8657,
    f1_macro_combined=0.859,
    f1_samples=0.9168,
)
validation_ax = plot_metric_comparison(
    {"SVM (dac6bb78)": svm_validation, labels[winner["variant"]]: result.metrics}
)
validation_ax.set_title("Validierung: SVM vs. bestes Sequenzmodell")
save_doc_figure(validation_ax.figure, "sequenzmodelle_validierung", MODEL_DOCS_FIGURES_DIR)

## Fazit

| | CV kombiniert | Val. Kultur | Val. Stadium | Val. kombiniert | Val. Macro-F1 kombiniert |
| --- | --- | --- | --- | --- | --- |
| SVM, alle Bänder (`dac6bb78`) | 0,859 | 0,920 | 0,913 | **0,866** | 0,859 |
| GRU, Sentinel-2 (`b6d2362b`) | 0,794 | 0,908 | 0,890 | 0,832 | 0,773 |
| bisher bestes Sequenzmodell (`combined_lstm_embeddings`, 70/30) | 0,711 | 0,812 | 0,793 | 0,700 | 0,613 |

- **Bestes Sequenzmodell im Projekt:** Die GRU auf 10 Sentinel-2-Bändern mit AEZ/Monat-Embeddings verbessert den bisherigen Bestwert auf der Validierung um 0,13 (0,700 → 0,832).
- **Die SVM bleibt das Endmodell:** Sie liegt auf der Validierung 0,034 vorn, beim Macro-F1 0,086. Mit rund 4.500 Trainingszeilen lohnt sich das Sequenzmodell nicht.
- **Validierung höher als CV** (0,832 gegenüber 0,794): Für die Validierung wird auf dem ganzen Trainingsteil trainiert, also mit 25 % mehr Zeilen als in einem CV-Fold. Das Netz profitiert davon stärker als die SVM.
- **Für M3:** Das Sequenzmodell profitiert aktiv von der Datenreduktion. Sentinel-2 ist hier nicht nur gleich gut, sondern besser als das volle Spektrum.
- **Für den hierarchischen Ansatz** (neues Issue): kleines Raster mit GRU und LSTM auf Sentinel-2, 8 Bänder pro Schritt für das volle Spektrum. Die Lernrate hat wenig Einfluss.